# HelpyHand Risk Scoring PoC
## 01 - EDA & Synthetic Data Design


# Business case

HelpyHand busca habilitar nano-créditos para poblaciones sin historial crediticio formal, utilizando modelos de riesgo construidos a partir de información alternativa. En un entorno real, esto implicaría integraciones con múltiples fuentes de datos externas. Sin embargo, para efectos de este Proof of Concept (PoC), construiremos un ecosistema de datos sintéticos que simule el comportamiento financiero de usuarios potenciales.

## Alcance de esta etapa

Este notebook constituye el primer paso en el desarrollo del motor de riesgo HelpyHand.

- Definir las tablas fuente del sistema
- Establecer granularidad
- Diseñar variables
- Definir relaciones
- Formular supuestos
- Crear prototipos
- Diseñar risk_features


# Dataset 1: Clients

Tabla central del sistema. Representa a cada cliente.

Granularidad: 1 fila = 1 cliente


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

N_CLIENTS = 20000

## client_id

In [ ]:
client_id = np.arange(1, N_CLIENTS + 1)

## national_id

In [ ]:
national_id = np.random.randint(10_000_000, 99_999_999, size=N_CLIENTS).astype(str)

## onboarding_date

In [ ]:
start_date = np.datetime64('2022-01-01')
end_date = np.datetime64('2024-12-31')

onboarding_date = start_date + np.random.randint(
    0, (end_date - start_date).astype(int), N_CLIENTS
).astype('timedelta64[D]')

## declared_income (Lognormal)

In [ ]:
mu = np.log(1_200_000)
sigma = 0.9

declared_income = np.random.lognormal(mean=mu, sigma=sigma, size=N_CLIENTS)
declared_income = np.clip(declared_income, 300_000, 15_000_000)

## employment_type

In [ ]:
employment_types = ['informal', 'formal', 'independent']
employment_probs = [0.55, 0.30, 0.15]

employment_type = np.random.choice(employment_types, size=N_CLIENTS, p=employment_probs)

## is_banked (dependiente de ingreso)

In [ ]:
income_scaled = declared_income / declared_income.max()
bank_prob = 0.2 + 0.6 * income_scaled

is_banked = np.random.rand(N_CLIENTS) < bank_prob

## Construcción final

In [ ]:
clients = pd.DataFrame({
    'client_id': client_id,
    'national_id': national_id,
    'onboarding_date': onboarding_date,
    'declared_income': declared_income.round(0),
    'employment_type': employment_type,
    'is_banked': is_banked
})

clients.head()

## Validaciones

In [ ]:
clients.describe()

In [ ]:
clients['employment_type'].value_counts(normalize=True)

# Estado actual

- Dataset clients generado correctamente
- Distribuciones realistas
- Aún no existe proceso generador de riesgo

## Próximos pasos

- Definir risk generating process
- Introducir correlaciones
- Construir siguientes tablas:
  - demographics
  - digital_sessions
  - transactions
  - payments
  - loan_applications
- Construir risk_features


# Risk Generating Process (RGP) Formal

## Definición del proceso generador de riesgo

El RGP se basa en un score latente `base_risk_score` por cliente, calculado como combinación lineal de factores heurísticos:

- `f_income`: inverso del ingreso logaritmizado
- `f_emp`: penalización por empleo informal/independiente
- `f_banked`: penalización por no bancarizado
- `f_age`: penalización por edades extremas
- `f_city`: penalización por zona rural
- `f_edu`: penalización por bajo nivel educativo

El score se normaliza y se usa para condicionar el comportamiento en otras tablas.

La probabilidad de default se calcula con logit: `p_default = sigmoid(intercept + coef * risk_index)`

## Implementación modular

Usamos módulos en `src/data_generation` para generar todas las tablas con correlaciones.

In [ ]:
# Importar módulos de generación
from src.data_generation import run_data_generation_pipeline

# Ejecutar pipeline completo
datasets = run_data_generation_pipeline(n_clients=5000, seed=42, base_rate=0.15, save_csv=True)

# Exploración de datos generados

In [ ]:
# Clients
print("Clients shape:", datasets["clients"].shape)
datasets["clients"].head()

In [ ]:
# Demographics
print("Demographics shape:", datasets["demographics"].shape)
datasets["demographics"].head()

In [ ]:
# Digital Sessions
print("Digital Sessions shape:", datasets["digital_sessions"].shape)
datasets["digital_sessions"].head()

In [ ]:
# Risk Features
print("Risk Features shape:", datasets["risk_features"].shape)
datasets["risk_features"].head()

In [ ]:
# Target
print("Target shape:", datasets["target"].shape)
print("Tasa de default:", datasets["target"]["default"].mean())
datasets["target"].head()

In [ ]:
# Análisis de monotonicidad: default vs income deciles
import numpy as np

df = datasets["target"].merge(datasets["clients"][["client_id", "declared_income"]], on="client_id")
df["income_decile"] = pd.qcut(df["declared_income"], 10, labels=False)
monotonicity = df.groupby("income_decile")["default"].mean()
print("Default rate por decil de ingreso:")
print(monotonicity)

In [1]:
import pandas as pd
df = pd.read_csv("../data/generated/model_mt.csv")
df.shape

(2000, 157)